# AI Energy Consumption Optimizer
## Phase 3: Preprocessing & Feature Engineering

This notebook will take the insights discovered in EDA, execute robust missing value interpolation, engineer our lag/rolling features, fit the scaling architecture directly onto the training split, and export them natively to the FastAPI-compatible `artifacts/` endpoints on Google Drive.

In [1]:
from google.colab import drive
import os
import pandas as pd
import numpy as np
import joblib
import json
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define standard base paths
BASE = '/content/drive/MyDrive/energy-optimizer'
RAW_PATH = os.path.join(BASE, 'data/raw/household_power_consumption.csv')

Mounted at /content/drive


In [2]:
# 3. Load & Resample Raw Dataset (From Phase 1/2 Baseline)
print("Mapping the Raw Payload & Initializing Base Form...")
df = pd.read_csv(RAW_PATH, sep=';', na_values=['?'], dtype={'Global_active_power': 'float64'}, nrows=None)
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
df.set_index('Datetime', inplace=True)
df.drop(columns=['Date', 'Time'], inplace=True)

df_hourly = df.resample('1H').mean()
df_hourly.rename(columns={'Global_active_power': 'consumption'}, inplace=True)
# We only need consumption for Tier 1 forecasting
df_hourly = df_hourly[['consumption']]
print(f"Initial Hourly Shape: {df_hourly.shape}")

Mapping the Raw Payload & Initializing Base Form...
Initial Hourly Shape: (34589, 1)


### Strategy: Missing Values
Forward-fill short gaps (`1-3h`). Linear Interpolate medium gaps (`4-24h`). Drop entire periods exceeding 24 hours.

In [3]:
# 4. Handling Missing Data (Continuous Series Engineering)

# Step A: Short gaps (<= 3h) ffill implies equipment lag or minor reboot
df_hourly['consumption'] = df_hourly['consumption'].ffill(limit=3)

# Step B: Moderate gaps (<= 24h) linear interpolation (temperature and daily cycles shift reasonably linearly)
df_hourly['consumption'] = df_hourly['consumption'].interpolate(method='linear', limit=24)

# Step C: Massive gaps removed (creates discontinuities, but preferable to hallucinated values)
before_drop = len(df_hourly)
df_hourly.dropna(inplace=True)
after_drop = len(df_hourly)

print(f"Dropped {before_drop - after_drop} unrecoverable missing hours.")
assert df_hourly.isnull().sum().sum() == 0, "CRITICAL ❌: Nulls still exist in the series."
print("✅ Missing values handled gracefully.")

Dropped 244 unrecoverable missing hours.
✅ Missing values handled gracefully.


### Strategy: Feature Engineering (No Data Leakage)
We must compute historic lags *on the full dataset* before splitting so that Validation and Test splits don't get `NaN` at their starting boundaries.

In [4]:
print("Executing Feature Engineering Engine...")
df_features = df_hourly.copy()

# A. Time-based Encoded Features
df_features['hour'] = df_features.index.hour
df_features['dayofweek'] = df_features.index.dayofweek
df_features['month'] = df_features.index.month
df_features['is_weekend'] = df_features['dayofweek'].apply(lambda x: 1 if x >= 5 else 0)

# Cyclical Encodings (Crucial for ML to understand 23:00 is close to 01:00)
df_features['hour_sin'] = np.sin(2 * np.pi * df_features['hour'] / 24)
df_features['hour_cos'] = np.cos(2 * np.pi * df_features['hour'] / 24)
df_features['month_sin'] = np.sin(2 * np.pi * df_features['month'] / 12)
df_features['month_cos'] = np.cos(2 * np.pi * df_features['month'] / 12)

# B. Safe Lag Features (Proven by Phase 2 PACF)
df_features['lag_1'] = df_features['consumption'].shift(1)
df_features['lag_2'] = df_features['consumption'].shift(2)
df_features['lag_24'] = df_features['consumption'].shift(24)
df_features['lag_48'] = df_features['consumption'].shift(48)
df_features['lag_168'] = df_features['consumption'].shift(168)

# C. Rolling Features (MUST use shift(1) so predicting 'now' uses 'past' data!)
df_features['rolling_mean_24h'] = df_features['consumption'].shift(1).rolling(window=24).mean()
df_features['rolling_std_24h'] = df_features['consumption'].shift(1).rolling(window=24).std()
df_features['rolling_mean_168h'] = df_features['consumption'].shift(1).rolling(window=168).mean()

# The first 168 rows are mathematically ruined by the lag. Drop them.
df_features.dropna(inplace=True)
print(f"✅ Features engineered. Final shape: {df_features.shape}")

Executing Feature Engineering Engine...
✅ Features engineered. Final shape: (34177, 17)


### Strategy: Temporal Split & Scaling
The test set must strictly follow the temporal order. Random shuffling causes massive data leakage in TS.

In [5]:
# 5. Strict Temporal Split (70 Train / 15 Val / 15 Test)
df_features.sort_index(inplace=True)
n = len(df_features)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df_features.iloc[:train_end].copy()
val_df = df_features.iloc[train_end:val_end].copy()
test_df = df_features.iloc[val_end:].copy()

print(f"Train ends at: {train_df.index[-1]}")
print(f"Val  starts at: {val_df.index[0]}")
print(f"Test starts at: {test_df.index[0]}")

# 6. Secure Scaling (FIT ON TRAIN ONLY)
# Determine scalable columns: Everything except our binary `is_weekend` and explicitly scaled `sin/cos` features
exclude_cols = ['is_weekend', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'hour', 'dayofweek', 'month']
scale_cols = [c for c in df_features.columns if c not in exclude_cols]

scaler = StandardScaler()
scaler.fit(train_df[scale_cols])

train_df[scale_cols] = scaler.transform(train_df[scale_cols])
val_df[scale_cols] = scaler.transform(val_df[scale_cols])
test_df[scale_cols] = scaler.transform(test_df[scale_cols])

print("✅ Standard Scaler FITTED and APPLIED securely via Train Isolation.")

Train ends at: 2009-09-18 00:00:00
Val  starts at: 2009-09-18 01:00:00
Test starts at: 2010-04-20 23:00:00
✅ Standard Scaler FITTED and APPLIED securely via Train Isolation.


### System Fix Alignment: Artifact Architecture Deployment
We must now output the dataset, scaler, feature alignments, and the **API Recent History Buffer** directly into your Google Drive's production folders.

In [6]:
# Output A: The Cleaned Datasets
train_df.to_csv(os.path.join(BASE, 'data/splits/train_scaled.csv'))
val_df.to_csv(os.path.join(BASE, 'data/splits/val_scaled.csv'))
test_df.to_csv(os.path.join(BASE, 'data/splits/test_scaled.csv'))
print("Stored Scaled Splits: /data/splits/")

# Output B: The API Scaler
scaler_path = os.path.join(BASE, 'artifacts/scalers/main_scaler.pkl')
joblib.dump(scaler, scaler_path)
print("Stored Production Scaler: /artifacts/scalers/")

# Output C: Ensure Feature Column Order for API Parsing
# The target y is dropped from this list because the API will only be fed the features (X) to make predictions.
target_feature_list = [c for c in df_features.columns if c != 'consumption']
with open(os.path.join(BASE, 'artifacts/scalers/feature_columns.json'), 'w') as f:
    json.dump(target_feature_list, f)
print("Stored Exact Feature Configuration Matrix!")

# Output D: THE RECENT HISTORY SIMULATION CACHE (Mandatory API Support)
# The API needs the last 168 unscaled 'raw' features from our processed data to act as its sliding memory base when the server first boots!
recent_history = df_features.tail(168)
recent_history.to_csv(os.path.join(BASE, 'data/processed/recent_history.csv'))
print("\n⭐ STORED STATIC RECENT HISTORY CACHE (168hr) in `data/processed`! This will bootstrap the FastAPI's internal memory buffer later.")

Stored Scaled Splits: /data/splits/
Stored Production Scaler: /artifacts/scalers/
Stored Exact Feature Configuration Matrix!

⭐ STORED STATIC RECENT HISTORY CACHE (168hr) in `data/processed`! This will bootstrap the FastAPI's internal memory buffer later.
